In [1]:
from data_model.DataCleaner import *
from dl_client import DatalakeClient
from data_model.manage_excel_support_file import *
from data_model.MergerTools import *
import pandas as pd
import os

client = DatalakeClient()
mergeTools = MergerTools()


In [6]:
real_path = 'cleaned/merged/combination/subMERGE_4-3_elecsys_FBP_fxd_0.csv'
generated_path = 'generated_synthetic_data/synthetic_subMERGE_4-3_elecsys_FBP_fxd_0.csv'
    

df_1 = client.download_file(real_path)
df_real = df_1.copy(deep=True)
metadata_real = client.get_metadata(real_path)
metadata_real = metadata_real['metadata']['custom']

df_2 = client.download_file(generated_path)
df_gen = df_2.copy(deep=True)
metadata_gen = client.get_metadata(generated_path)
metadata_gen = metadata_gen['metadata']['custom']

In [7]:
# Calcola il numero di visite per ciascun soggetto (RID)
visite_per_soggetto = df_real.groupby('RID').size()
print(visite_per_soggetto.mean())
print(visite_per_soggetto.median())
print(visite_per_soggetto.min())
print(visite_per_soggetto.max())




3.3738202708247846
3.0
1
15


In [8]:
# Calcola il numero di visite per ciascun soggetto (RID)
visite_per_soggetto_gen = df_gen.groupby('RID').size()
print(visite_per_soggetto_gen.mean())
print(visite_per_soggetto_gen.median())
print(visite_per_soggetto_gen.min())
print(visite_per_soggetto_gen.max())

4.499242424242424
4.0
1
12


In [14]:
missing_columns = [col for col in df_real.columns if col not in df_gen.columns]
print("Colonne presenti in df_real ma mancanti in df_gen:", missing_columns)

Colonne presenti in df_real ma mancanti in df_gen: ['COHORT', 'VISCODE', 'EXAMDATE', 'FLDSTRENG', 'FSVERSION', 'IMAGEUID', 'update_stamp', 'STATUS', 'METHOD_CSF', 'METHOD_PET', 'TRACER']


## 1 - rinomina variabili

In [ ]:
# Rinomina le colonne ID -> RID e TIME -> AGE
df_gen = df_gen.rename(columns={'ID': 'RID', 'TIME': 'AGE'})
df_gen

## 2 - VISIT_MONTH
Calcolo e aggiungo la variabile VISIT_MONTH a partire dall'età

In [ ]:
# Calcola VISIT_MONTH per ciascun soggetto
# Ordina per RID e AGE per assicurarsi che le visite siano in ordine cronologico
df_gen = df_gen.sort_values(['RID', 'AGE'])

# Raggruppa per RID e calcola la differenza dall'AGE della prima visita
# AGE è in anni, quindi moltiplico per 12 per ottenere i mesi e arrotondo all'intero
df_gen['VISIT_MONTH'] = df_gen.groupby('RID')['AGE'].transform(
    lambda x: ((x - x.iloc[0]) * 12).round().astype(int)
)


In [ ]:
# Riordina le colonne per mettere VISIT_MONTH prima di AGE
cols = df_gen.columns.tolist()
cols.remove('VISIT_MONTH')
cols.insert(2, 'VISIT_MONTH')
df_gen = df_gen[cols]

In [ ]:
df_gen

## 3 - Ri Dummies
Trasformo nuovamente in Dummies le variabili categoriche

In [ ]:
df_gen.columns

In [ ]:
# Verifica la presenza di NaN nelle colonne specificate
colonne_da_verificare = ['GENDER', 'DX', 'MARRY', 'ETHNICITY', 'RACE']  

for col in colonne_da_verificare:
    if col in df_gen.columns:
        n_nan = df_gen[col].isna().sum()
        print(f"{col}: {n_nan} NaN ({n_nan/len(df_gen)*100:.2f}%)")
    else:
        print(f"{col}: colonna non trovata")


In [ ]:
# Crea dummy variables per le colonne categoriche mantenendo i NaN
colonne_dummy = ['GENDER', 'DX', 'MARRY', 'ETHNICITY', 'RACE']

df_gen = pd.get_dummies(df_gen, columns=colonne_dummy, prefix_sep='/', dummy_na=False, dtype=int)

# Per ogni colonna originale che aveva NaN, imposta NaN in tutte le sue dummy corrispondenti
for col in colonne_dummy:
    # Trova le colonne dummy create per questa colonna
    dummy_cols = [c for c in df_gen.columns if c.startswith(f"{col}/")]
    
    # Identifica le righe dove la colonna originale era NaN
    # (tutte le dummy di quella riga saranno 0)
    mask = df_gen[dummy_cols].sum(axis=1) == 0
    
    # Imposta NaN per tutte le dummy di quelle righe
    df_gen.loc[mask, dummy_cols] = pd.NA

In [ ]:
df_gen[['GENDER/female', 'GENDER/male', 'DX/CN', 'DX/Dementia', 'DX/MCI',
       'MARRY/divorced', 'MARRY/married', 'MARRY/single', 'MARRY/widowed',
       'ETHNICITY/latino', 'ETHNICITY/not_latino', 'RACE/Asian', 'RACE/Black',
       'RACE/Mixed', 'RACE/Native_american', 'RACE/White']].isna().sum()

## 4 - Aggiorno i metadati 
aggiungendo i metadati del file reale e cambio il file_code

In [ ]:
metadata_gen['file_code'] = 'synthetic_subMERGE_4-3_elecsys_FBP_fxd_0'
metadata_gen['level'] = 'synthetic'
metadata_gen['generation_date'] = '2026-01-15T12:48:00'

In [ ]:
colonne_da_aggiungere = [col for col in metadata_real.keys() if col not in metadata_gen.keys() and col != 'dummy_fix_date']

for col in colonne_da_aggiungere:
    metadata_gen[col] = metadata_real[col]

In [ ]:
metadata_gen

## 5 - Salvo
Salvataggio del file con nuovo nome in nuova posizione (per evitare sia mai sovrascritto)

In [ ]:
result = client.upload_dataframe(
    df=df_gen,
    object_name='synthetic_subMERGE_4-3_elecsys_FBP_fxd_0.csv',
    prefix='generated_synthetic_data',
    metadata=metadata_gen
)